In [40]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import ssl
import pandas as pd

In [41]:
df = pd.read_csv("GoEmotions/data/train.tsv", sep="\t", header=None)
df.to_csv("GoEmotions_train.csv", index=False)

df = pd.read_csv("GoEmotions/data/test.tsv", sep="\t", header=None)
df.to_csv("GoEmotions_test.csv", index=False)

with open("GoEmotions/data/emotions.txt") as f:
    emotions = [line.strip() for line in f]

df = pd.DataFrame({"emotion": emotions})
df.to_csv("emotions.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'GoEmotions/data/train.tsv'

In [42]:
import os

raw_dir = "SemEvalDataset"
out_dir = "SemEvalDataset/processed"

os.makedirs(out_dir, exist_ok=True)

files = {
    "train": "2018-E-c-En-train.txt",
    "dev": "2018-E-c-En-dev.txt",
    "test": "2018-E-c-En-test.txt"
}

for split, filename in files.items():
    path = os.path.join(raw_dir, filename)

    df = pd.read_csv(path, sep="\t")

    output_path = os.path.join(out_dir, f"{split}.csv")
    df.to_csv(output_path, index=False)

    print(f"Saved {output_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'SemEvalDataset/2018-E-c-En-train.txt'

# Loading and Preprocessing:

In [43]:
G_train = pd.read_csv("GoEmotions/Processed/GoEmotions_train.csv")
G_test = pd.read_csv("GoEmotions/Processed/GoEmotions_test.csv")

S_train = pd.read_csv("SemEvalDataset/processed/train.csv")
S_test = pd.read_csv("SemEvalDataset/processed/test.csv")

T_train = pd.read_csv("TwitterEmotions/training.csv")
T_test = pd.read_csv("TwitterEmotions/test.csv")

In [45]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return re.findall(r"\b[a-z]+\b", clean_text(text))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def pad_tokens(tokens, max_len=100, padding="post"):
    tokens = tokens[:max_len]
    pad_amount = max_len - len(tokens)

    if padding == "post":
        return tokens + ["<PAD>"] * pad_amount
    return ["<PAD>"] * pad_amount + tokens

In [46]:
def preprocess_only(df, text_col, max_len=100):
    df = df.copy()

    df["clean_text"] = df[text_col].apply(clean_text)

    df["tokens"] = df[text_col].apply(tokenize_text)

    df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

    df["padded_tokens"] = df["tokens_no_stopwords"].apply(
        lambda tokens: pad_tokens(tokens, max_len=max_len, padding="post")
    )

    df["word_count"] = df["tokens"].apply(len)
    df["clean_word_count"] = df["tokens_no_stopwords"].apply(len)
    df["char_count"] = df["clean_text"].apply(len)

    scaler = StandardScaler()
    df[["word_count_scaled", "clean_word_count_scaled", "char_count_scaled"]] = scaler.fit_transform(
        df[["word_count", "clean_word_count", "char_count"]]
    )

    return df

In [48]:
G_train = G_train.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})
G_test = G_test.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})
print(T_train.columns)
print(S_train.columns)
print(G_train.columns)

Index(['text', 'label'], dtype='str')
Index(['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy',
       'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'],
      dtype='str')
Index(['text', 'label_id', 'example_id'], dtype='str')


In [49]:
twitter_train = preprocess_only(T_train, text_col="text", max_len=100)
twitter_test = preprocess_only(T_test, text_col="text", max_len=100)

semeval_train = preprocess_only(S_train, text_col="Tweet", max_len=100)
semeval_test = preprocess_only(S_test, text_col="Tweet", max_len=100)

goemotions_train = preprocess_only(G_train, text_col="text", max_len=100)
goemotions_test = preprocess_only(G_test, text_col="text", max_len=100)

In [ ]:
emotions = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism", "pride",
    "realization", "relief", "remorse", "sadness", "surprise", "neutral"
]

def parse_labels(label_str):
    return [int(x) for x in str(label_str).split(",")]

def make_multilabel_columns(df):
    df = df.copy()

    # Convert "13,18" into [13, 18]
    df["label_list"] = df["label_id"].apply(parse_labels)

    # Add readable labels: [13,18] -> ["excitement", "love"]
    df["label_names"] = df["label_list"].apply(
        lambda ids: [emotions[i] for i in ids]
    )

    # Create one binary column per emotion
    for i, emotion in enumerate(emotions):
        df[emotion] = df["label_list"].apply(lambda labels: 1 if i in labels else 0)

    return df

goemotions_train = make_multilabel_columns(goemotions_train)
goemotions_test = make_multilabel_columns(goemotions_test)

In [53]:
print(goemotions_train[["label_id", "label_list", "label_names"]].head())

emotion_cols = emotions
print(goemotions_train[emotion_cols].head())

  label_id label_list  label_names
0       27       [27]    [neutral]
1       27       [27]    [neutral]
2        2        [2]      [anger]
3       14       [14]       [fear]
4        3        [3]  [annoyance]
   admiration  amusement  anger  annoyance  approval  caring  confusion  \
0           0          0      0          0         0       0          0   
1           0          0      0          0         0       0          0   
2           0          0      1          0         0       0          0   
3           0          0      0          0         0       0          0   
4           0          0      0          1         0       0          0   

   curiosity  desire  disappointment  ...  love  nervousness  optimism  pride  \
0          0       0               0  ...     0            0         0      0   
1          0       0               0  ...     0            0         0      0   
2          0       0               0  ...     0            0         0      0   
3          0   

In [54]:
print(goemotions_test.columns)
print(goemotions_train.columns)

Index(['text', 'label_id', 'example_id', 'clean_text', 'tokens',
       'tokens_no_stopwords', 'padded_tokens', 'word_count',
       'clean_word_count', 'char_count', 'word_count_scaled',
       'clean_word_count_scaled', 'char_count_scaled', 'label', 'label_list',
       'label_names', 'admiration', 'amusement', 'anger', 'annoyance',
       'approval', 'caring', 'confusion', 'curiosity', 'desire',
       'disappointment', 'disapproval', 'disgust', 'embarrassment',
       'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
       'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse',
       'sadness', 'surprise', 'neutral'],
      dtype='str')
Index(['text', 'label_id', 'example_id', 'clean_text', 'tokens',
       'tokens_no_stopwords', 'padded_tokens', 'word_count',
       'clean_word_count', 'char_count', 'word_count_scaled',
       'clean_word_count_scaled', 'char_count_scaled', 'label', 'label_list',
       'label_names', 'admiration', 'amusement', 'anger', 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_text = vectorizer.fit_transform(
    goemotions_train["clean_text"]
)

X_test_text = vectorizer.transform(
    goemotions_test["clean_text"]
)

from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

metadata_cols = ["clean_word_count", "char_count"]

scaler = StandardScaler()

X_train_meta = scaler.fit_transform(goemotions_train[metadata_cols])
X_test_meta = scaler.transform(goemotions_test[metadata_cols])

X_train = hstack([X_train_text, csr_matrix(X_train_meta)])
X_test = hstack([X_test_text, csr_matrix(X_test_meta)])

from scipy.sparse import hstack

X_train = hstack([X_train_text, X_train_meta])
Y_train = goemotions_train[emotions]
X_test = hstack([X_test_text, X_test_meta])
Y_test = goemotions_test[emotions]